In [2]:
import shutil
from datasets import load_dataset

free_gb = shutil.disk_usage("/").free / 1e9
print(f"Free disk space: {free_gb:.1f} GB")
assert free_gb > 5, "Under 5GB free — clear space before continuing (see huggingface-cli delete-cache)"

hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    split="validation[:200]",
    revision="refs/convert/parquet",
)

nq = load_dataset(
    "google-research-datasets/nq_open",
    split="validation[:200]",
)

print("HotpotQA:", len(hotpot), "examples")
print("NQ:", len(nq), "examples")
print("\nHotpotQA sample:", hotpot[0])
print("\nNQ sample:", nq[0])

Free disk space: 8.1 GB
HotpotQA: 200 examples
NQ: 200 examples

HotpotQA sample: {'id': '5a8b57f25542995d1e6f1371', 'question': 'Were Scott Derrickson and Ed Wood of the same nationality?', 'answer': 'yes', 'type': 'comparison', 'level': 'hard', 'supporting_facts': {'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}, 'context': {'title': ['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', 'Deliver Us from Evil (2014 film)', 'Adam Collis', 'Sinister (film)', 'Conrad Brooks', 'Doctor Strange (2016 film)'], 'sentences': [['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.', " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.", ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cas

In [3]:
import sys
sys.path.append("..")  # so we can import from the project root

import random
import json
from perturbations.generators import (
    inject_contradiction,
    inject_lexical_distractor,
    inject_stale_document,
    inject_multihop_gap,
    inject_topical_noise,
)

rng = random.Random(42)  # fixed seed for reproducibility
hotpot_list = list(hotpot)  # convert HF Dataset to plain list of dicts, easier to work with

PERTURBATIONS = {
    "contradiction": inject_contradiction,
    "lexical_distractor": inject_lexical_distractor,
    "stale_document": inject_stale_document,
    "multihop_gap": inject_multihop_gap,
    "topical_noise": inject_topical_noise,
}

stress_test_suite = []
for example in hotpot_list:
    for name, fn in PERTURBATIONS.items():
        perturbed = fn(example, corpus_pool=hotpot_list, rng=rng)
        stress_test_suite.append(perturbed)

print(f"Built {len(stress_test_suite)} stress-test examples "
      f"({len(hotpot_list)} originals x {len(PERTURBATIONS)} perturbation types)")

# Quick sanity check: print one example per perturbation type
seen_types = set()
for ex in stress_test_suite:
    if ex["perturbation_type"] not in seen_types:
        seen_types.add(ex["perturbation_type"])
        print(f"\n=== {ex['perturbation_type']} ===")
        print("Question:", ex["question"])
        print("Answer:", ex["answer"])
        print("Context titles:", ex["context"]["title"])

# Save to disk so later phases don't need to regenerate this
with open("../data/datasets/stress_test_suite.json", "w") as f:
    json.dump(stress_test_suite, f, indent=2)
print("\nSaved to data/datasets/stress_test_suite.json")

Built 1000 stress-test examples (200 originals x 5 perturbation types)

=== contradiction ===
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Context titles: ['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', 'Deliver Us from Evil (2014 film)', 'Adam Collis', 'Sinister (film)', 'Conrad Brooks', 'Doctor Strange (2016 film)']

=== lexical_distractor ===
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Context titles: ['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', 'Deliver Us from Evil (2014 film)', 'Adam Collis', 'Sinister (film)', 'Conrad Brooks', 'Doctor Strange (2016 film)', 'Gone in 60 Seconds (2000 film) (lexical distractor)']

=== stale_document ===
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Context titles: ['Ed Wood (film)', 'Scott Derrickson (2010 archive)', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', '

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # downloads once, ~90MB, cached locally


def build_local_index(example: dict):
    """Build a small FAISS index over just this example's candidate passages."""
    passages = ["".join(sents) for sents in example["context"]["sentences"]]
    titles = example["context"]["title"]

    embeddings = embed_model.encode(passages, convert_to_numpy=True, normalize_embeddings=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product on normalized vectors = cosine similarity
    index.add(embeddings)
    return index, titles, passages


def retrieve_top_k(example: dict, k: int = 3):
    """Embed the question, search this example's local index, return top-k (title, passage, score)."""
    index, titles, passages = build_local_index(example)
    query_vec = embed_model.encode([example["question"]], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec, k)
    return [(titles[i], passages[i], float(scores[0][j])) for j, i in enumerate(indices[0])]


# Sanity check: run retrieval on the first contradiction example and inspect
sample = stress_test_suite[0]
print("Question:", sample["question"])
print("Gold titles:", sample["supporting_facts"]["title"])
print("\nTop-3 retrieved:")
for title, passage, score in retrieve_top_k(sample, k=3):
    print(f"  [{score:.3f}] {title}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Avahan  Masih\Desktop\RAG-Diagnostic-Suite\RAG-Diagnostic-Suite\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Avahan  Masih\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold titles: ['Scott Derrickson', 'Ed Wood']

Top-3 retrieved:
  [0.503] Scott Derrickson
  [0.494] Ed Wood (film)
  [0.486] Ed Wood
